# Notebook de definición del problema y entorno reproducible — Fase 1

**Proyecto transversal · MCDI500 Programación para la Ciencia de Datos**
**Magíster en Ciencia de Datos e Inteligencia Artificial · Universidad Andrés Bello**

**Grupo 5 — Factores asociados a la duración autorizada de los permisos de uso de vía
pública en San Francisco**

## Introducción

Este cuaderno deja establecido lo que la Fase 2 necesita para empezar: la definición del
problema, un entorno verificado, la estructura del repositorio con sus artefactos de
reproducibilidad, un módulo propio importable y la documentación del conjunto de datos con su
evaluación contra los criterios del curso.

No transforma datos: los define. La limpieza, la imputación y el escalamiento son trabajo de
la Fase 2, que lee lo que aquí se decide en lugar de volver a decidirlo.

**Orden de ejecución.** Lineal, de arriba abajo. *Kernel → Restart Kernel and Run All Cells*
debe terminar sin errores y con la numeración de ejecución continua.

### Herramientas del ecosistema científico

Cada importación responde a una necesidad concreta de esta fase y no se incluye ninguna que
no se use. `pathlib` resuelve rutas de forma independiente del sistema operativo; `subprocess`
y `shutil` consultan el estado real de Git sin suponer que esté instalado; `json` exporta los
metadatos en un formato legible tanto por una persona como por el cuaderno de la Fase 2.

In [1]:
import sys                       # intérprete en uso: es lo que se verifica más abajo
import json                      # exportación de metadatos legibles
import platform                  # sistema operativo, para dejarlo en la bitácora
import subprocess                # consultas a Git desde el cuaderno
import shutil                    # localizar ejecutables (git) sin suponer que existen
import importlib                 # importar el módulo que este cuaderno va a escribir
import warnings                  # capturar los avisos de parseo como evidencia, no como ruido
import io, contextlib            # capturar salidas para los anexos del informe
from pathlib import Path         # manejo de rutas independiente del sistema operativo
from datetime import date

import numpy as np               # presente para verificar el entorno del proyecto
import pandas as pd              # tablas de documentación de la fase

# Reproducibilidad: la misma semilla que usa el cuaderno de la Fase 2.
SEMILLA = 42
np.random.seed(SEMILLA)

def capturar(funcion, *args, **kwargs):
    """
    Ejecuta una funcion, muestra su salida en pantalla y ademas la devuelve como texto.

    Sirve para que los anexos del informe se generen desde la misma ejecucion que produce
    los resultados, en lugar de copiarse a mano desde una captura de pantalla.

    Retorna
    -------
    (obj, str)
        Lo que devuelva la funcion y su salida estandar como texto.
    """
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        resultado = funcion(*args, **kwargs)
    texto = buffer.getvalue()
    print(texto, end="")
    return resultado, texto


print("Cuaderno de la Fase 1 ·", date.today().isoformat())

Cuaderno de la Fase 1 · 2026-09-15


## 1. Definición del problema

Toda la identificación del proyecto vive en un único diccionario. Lo que el cuaderno imprima
o persista después se deriva de aquí, de modo que no circulen dos versiones del título, de la
pregunta o de los integrantes.

> **Completar antes de entregar:** los nombres de los cuatro integrantes. El correo
> configurado en Git debe coincidir con el de su cuenta de GitHub, o los *commits* no se les
> atribuyen.

In [2]:
PROYECTO = {
    "titulo": "Factores asociados a la duración autorizada de los permisos de uso de vía "
              "pública en San Francisco",
    "grupo": "Grupo 5",
    "asignatura": "MCDI500 · Programación para la Ciencia de Datos",
    "integrantes": [
        "MARCO ÁLVAREZ ARAYA (@Marco30101994-Unab)",
        "MARÍA FASSLER NEUMANN (@belenfassler)",
        "RICARDO JARAMILLO PULGAR (@RJaramilloP)"],
    "problematica": (
        "Los permisos de uso de vía pública regulan la ocupación temporal del espacio público "
        "por excavaciones, obras, instalaciones de telecomunicaciones y usos comerciales. La "
        "duración autorizada de cada permiso determina cuánto tiempo una calle queda afectada. "
        "El problema es de asociación: identificar qué factores se relacionan con esa duración, "
        "medida como el número de días entre la fecha de inicio y la de término del permiso."
    ),
    "objetivo_general": (
        "Identificar y cuantificar los factores asociados a la duración autorizada de los "
        "permisos de uso de vía pública vigentes en San Francisco, mediante un flujo "
        "reproducible de obtención, saneamiento y validación de datos abiertos."
    ),
    "objetivos_especificos": [
        "Definir el problema y establecer el entorno reproducible del proyecto (F1).",
        "Documentar la procedencia, la licencia y el rol analítico de cada variable (F1).",
        "Construir el pipeline de obtención, limpieza y transformación de datos (F2).",
        "Implementar el núcleo algorítmico con programación estructurada y POO (F3).",
        "Comunicar los hallazgos mediante visualización y un informe técnico (F4).",
    ],
    "preguntas": [
        "¿Qué proporción de la variación en la duración se asocia al tipo de permiso?",
        "Dentro de un mismo tipo, ¿el barrio y el agente solicitante distinguen duraciones?",
        "¿Qué variables concentran los faltantes y qué significa que un dato no esté?",
    ],
    "criterios_exito": [
        "El repositorio se clona y ambos cuadernos corren completos sin intervención manual.",
        "Cada decisión de preprocesamiento queda justificada con cifras, no con adjetivos.",
        "Los cuatro integrantes tienen commits propios en el historial.",
    ],
    "alcance": {
        "incluye": ["Definición del problema", "Entorno reproducible",
                    "Selección y documentación del conjunto"],
        "excluye": ["Limpieza e imputación", "Modelado", "Inferencia causal"],
    },
    "limitaciones": [
        "Los datos son secundarios: no se controló su recolección ni su representatividad.",
        "La fuente publica solo permisos vigentes, de modo que los de mayor duración quedan "
        "sobrerrepresentados. Se cuantifica en la sección 5.4.",
        "El portal se actualiza a diario: los resultados corresponden a un corte concreto y no "
        "son reproducibles descargando el archivo de nuevo.",
    ],
}

La función siguiente presenta el proyecto en pantalla. Recibe el diccionario como
parámetro en lugar de leer una variable global, de modo que pueda reutilizarse y probarse con
cualquier configuración. Falla temprano y con un mensaje que dice **qué** falta, no solo que
algo salió mal.

In [3]:
def presentar_proyecto(config):
    """
    Imprime la definicion del proyecto de forma legible.

    Parametros
    ----------
    config : dict
        Diccionario de configuracion del proyecto.

    Retorna
    -------
    None

    Lanza
    -----
    KeyError
        Si falta alguna de las claves obligatorias.
    """
    obligatorias = ("titulo", "problematica", "objetivo_general", "objetivos_especificos")
    faltantes = [c for c in obligatorias if c not in config]
    if faltantes:
        # Se falla temprano y con un mensaje que dice QUE falta, no solo que algo fallo.
        raise KeyError(f"Faltan claves obligatorias en la configuracion: {faltantes}")

    print(config["titulo"].upper())
    print("=" * 60)
    print(f"{config['asignatura']} · {config['grupo']}")
    print("\nProblemática")
    print(config["problematica"])
    print("\nObjetivo general")
    print(config["objetivo_general"])
    print("\nObjetivos específicos")
    for i, obj in enumerate(config["objetivos_especificos"], start=1):
        print(f"  {i}. {obj}")
    print("\nPreguntas que orientan el trabajo")
    for pregunta in config["preguntas"]:
        print(f"  · {pregunta}")


presentar_proyecto(PROYECTO)

FACTORES ASOCIADOS A LA DURACIÓN AUTORIZADA DE LOS PERMISOS DE USO DE VÍA PÚBLICA EN SAN FRANCISCO
MCDI500 · Programación para la Ciencia de Datos · Grupo 5

Problemática
Los permisos de uso de vía pública regulan la ocupación temporal del espacio público por excavaciones, obras, instalaciones de telecomunicaciones y usos comerciales. La duración autorizada de cada permiso determina cuánto tiempo una calle queda afectada. El problema es de asociación: identificar qué factores se relacionan con esa duración, medida como el número de días entre la fecha de inicio y la de término del permiso.

Objetivo general
Identificar y cuantificar los factores asociados a la duración autorizada de los permisos de uso de vía pública vigentes en San Francisco, mediante un flujo reproducible de obtención, saneamiento y validación de datos abiertos.

Objetivos específicos
  1. Definir el problema y establecer el entorno reproducible del proyecto (F1).
  2. Documentar la procedencia, la licencia y el rol 

El alcance y las limitaciones se presentan aparte porque cumplen una función distinta:
el primero delimita qué entra en esta entrega y qué no; las segundas declaran qué no puede
afirmarse con estos datos, por bien que se haga el trabajo.

In [4]:
print("Alcance de la Fase 1")
print("  Incluye:", ", ".join(PROYECTO["alcance"]["incluye"]))
print("  Excluye:", ", ".join(PROYECTO["alcance"]["excluye"]))

print("\nLimitaciones declaradas")
for lim in PROYECTO["limitaciones"]:
    print(f"  · {lim}")

Alcance de la Fase 1
  Incluye: Definición del problema, Entorno reproducible, Selección y documentación del conjunto
  Excluye: Limpieza e imputación, Modelado, Inferencia causal

Limitaciones declaradas
  · Los datos son secundarios: no se controló su recolección ni su representatividad.
  · La fuente publica solo permisos vigentes, de modo que los de mayor duración quedan sobrerrepresentados. Se cuantifica en la sección 5.4.
  · El portal se actualiza a diario: los resultados corresponden a un corte concreto y no son reproducibles descargando el archivo de nuevo.


> **Sobre las limitaciones.** Declararlas antes de haber analizado nada es señal de haber
> entendido la fuente, no de desconfianza en el trabajo. La segunda de ellas —la
> sobrerrepresentación de los permisos largos— no es una sospecha: la sección 5.4 la mide y
> muestra que la duración mínima observada crece con la antigüedad del permiso, que es
> exactamente lo que predice el mecanismo.

## 2. Verificación del entorno

Se registra el entorno **realmente en uso**, no el que se esperaba tener. Si alguna versión
difiere entre integrantes, esta salida lo revela antes de que se convierta en un resultado
distinto sin explicación.

**Cómo se lee este resultado.** Cada línea corresponde a un acoplamiento que la
reproducibilidad exige explicitar. Si el intérprete no apunta al `.venv` del proyecto, el
cuaderno está corriendo con las librerías del sistema y las versiones que se registren más
abajo no serán las que el grupo declaró. Si alguna librería aparece como `[FALTA]`, el
cuaderno no se interrumpe: informa qué instalar, porque detenerse obligaría a reejecutar todo
desde el principio por un problema que se resuelve en una línea.

## 3. Estructura del repositorio y artefactos

Las rutas de datos son **comunes a todas las fases**: son exactamente las que usa el cuaderno
de la Fase 2, de modo que un mismo archivo sirva a los dos. Cada fase tiene su propia carpeta
para cuadernos e informes.